In [ ]:
# Merge the prediction results of 12 clusters into a single file (12 csv files → 1 csv file)
import os
import pandas as pd

# Specify the input folder path and output folder path
input_folder_path = 'D:\DESKTOP\desk\\bianliang\esm-ssp585\p12'
output_file_path = 'D:\DESKTOP\desk\\bianliang\esm-ssp585\p12\p12ALL.csv'

combined_df = pd.DataFrame()

for filename in os.listdir(input_folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder_path, filename)
        
        df = pd.read_csv(file_path)
        
        combined_df = pd.concat([combined_df, df], ignore_index=True)

combined_df.to_csv(output_file_path, index=False)

print(f"Successfully merged the contents of all CSV files, the result is saved in '{output_file_path}'.")

In [ ]:
# Calculate the annual mean trend of global hydrological and runoff variables (for line chart plotting); P, ET, R, n
import pandas as pd
import os

# Input file path
input_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\p12\\p12ALL.csv'

# Output file path
output_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\R-trend\\p12ALL-trend-585.csv'

# Create output directory if it does not exist
output_dir = os.path.dirname(output_file)
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_file)

# List of target variables
variables = ['PPT', 'PET', 'R', 'n', 'AET']

# Initialize dataframe for weighted results
weighted_results = pd.DataFrame()

# Calculate area-weighted annual mean for each variable
weighted_results['year'] = df['year'].unique()
for var in variables:
    weighted_var = df.groupby('year').apply(lambda x: (x[var] * x['AREA']).sum() / x['AREA'].sum()).reset_index(name=var)
    weighted_results = pd.merge(weighted_results, weighted_var, on='year')

# Save the weighted results to file
weighted_results.to_csv(output_file, index=False)

print("Weighted results have been saved to:", output_file)

In [ ]:
# Global‑average attribution trend
import pandas as pd
import numpy as np
import math

# Define Choudhury‑Yang Budyko model function
def budyko(PPT, PET, n):
    phi = PET / PPT
    ET = PPT * (1 + phi - (1 + phi**n)**(1/n))
    R = PPT - ET
    return R

# Define elasticity coefficient calculation function
def compute_elasticities(PPT, PET,  n):
    phi = PET / PPT
    # Analytically derived elasticity coefficients
    eps_ppt =((1+phi**n)**(1/n+1)-phi**(n+1))/((1+phi**n)*((1+phi**n)**(1/n)-phi))
    eps_pet =1/((1+phi**n)*((1-(1+phi**-n)**(1/n))))
    eps_n =(math.log(1+phi**n)+phi**n*math.log(1+phi**-n))/(n*(1+phi**n)*(1-(1+phi**-n)**(1/n)))
    return eps_ppt, eps_pet, eps_n

# Read input data
input_file = 'D:\DESKTOP\desk\\bianliang\esm-ssp585-ssp126Lu\\attribution\guiyin\p12ALL-trend-585.csv'
output_file = 'D:\DESKTOP\desk\\bianliang\esm-ssp585-ssp126Lu\\attribution\guiyin\esm-ssp585-out.csv'

df = pd.read_csv(input_file)

# Full time period: 2015‑2100
all_period = df[(df['year'] >= 2015) & (df['year'] <= 2100)]

# Averaging over the full period
PPT00 = all_period['PPT'].mean()
PET00 = all_period['PET'].mean()
n00 = all_period['n'].mean()
R00 = all_period['R'].mean()

# Calculate elasticity coefficients for the full period
eps_ppt00, eps_pet00, eps_n00 = compute_elasticities(PPT00, PET00, n00)

# Select baseline period: 1985‑2014
base_period = df[(df['year'] >= 1985) & (df['year'] <= 2014)]

# Averaging over the baseline period
PPT0 = base_period['PPT'].mean()
PET0 = base_period['PET'].mean()
n0 = base_period['n'].mean()
R0 = base_period['R'].mean()

# Calculate elasticity coefficients for the baseline period
eps_ppt0, eps_pet0, eps_n0 = compute_elasticities(PPT0, PET0, n0)

# Initialize result list
results = []

# Changing period: 30‑year sliding window starting from 1986
start_year = 1986
end_year = df['year'].max()

while start_year + 30 - 1 <= end_year:
    period = df[(df['year'] >= start_year) & (df['year'] <= start_year + 30 - 1)]
    
    PPT1 = period['PPT'].mean()
    PET1 = period['PET'].mean()
    n1 = period['n'].mean()
    R1 = period['R'].mean()

    # Elasticity coefficients for the changing period
    eps_ppt1, eps_pet1, eps_n1 = compute_elasticities(PPT1, PET1, n1)

    # Attribution using elasticity coefficients from the full period
    delta_PPT = PPT1 - PPT0
    delta_PET = PET1 - PET0
    delta_n = n1 - n0

    delta_RPPT = delta_PPT / PPT00 * eps_ppt00 * R00
    delta_RPET = delta_PET / PET00 * eps_pet00 * R00
    delta_Rn = delta_n / n00 * eps_n00 * R00
    delta_RCC = delta_RPPT + delta_RPET
    delta_R_total = delta_RPPT + delta_RPET + delta_Rn

    # Contribution rate (percentage)
    if delta_R_total != 0:
        delta_PPT_percent = delta_RPPT / delta_R_total * 100
        delta_PET_percent = delta_RPET / delta_R_total * 100
        delta_n_percent = delta_Rn / delta_R_total * 100
        delta_CC_percent = delta_PPT_percent + delta_PET_percent
    else:
        delta_PPT_percent = delta_PET_percent = delta_n_percent = np.nan

    results.append([
        start_year + 30 - 1,  # Last year of sliding window
        #eps_ppt00, eps_pet00, eps_n00,    # Elasticity for full period
        #eps_ppt0, eps_pet0, eps_n0,    # Elasticity for baseline period
        eps_ppt1, eps_pet1, eps_n1,
        delta_RPPT, delta_RPET,delta_RCC, delta_Rn, delta_R_total,
        delta_PPT_percent, delta_PET_percent,delta_CC_percent, delta_n_percent
    ])

    # Slide window forward by one year
    start_year += 1

# Convert to DataFrame and save
# esp_xx: elasticity; dRxx: runoff change induced by each factor (ΔRxx); pxx: contribution percentage of each factor
columns = [
    'Year', 
    #'εPPT00', 'εPET00', 'εn00',
    #'εPPT0', 'εPET0', 'εn0',
    'eps_PPT', 'eps_PET', 'eps_n',
    'dRPPT', 'dRPET', 'dRCC', 'dRn', 'dR',
    'pPPT', 'pPET', 'pCC', 'pn'
]
output_df = pd.DataFrame(results, columns=columns)
output_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Calculation completed! Results saved to {output_file}")